# RCR: Poway-Midland Railroad Digital Experience
## Technical Blog — Capstone Project

This blog documents how our team (Rebecca, Cyrus, Rishabh) applied each required technical area in our AP CSP capstone project. Our goal was to transform the [Poway-Midland Railroad](https://powaymidlandrr.org)'s static informational website into a connected, data-driven visitor platform.

**Stack:** Flask + SQLite backend · Jekyll frontend · scikit-learn ML · Deployed via GitHub Pages

---

## 1. Modern UI/UX

### What We Built
Every page of the site uses a unified dark railroad theme built entirely with CSS custom properties (variables). The design system uses warm coal and rust tones that match the aesthetic of a historic steam railroad, while remaining clean and readable.

**Key design decisions:**
- CSS variables for consistent theming across 14+ pages
- Fixed navigation header with dropdown menus and user avatar state
- Animated train background on the home page (pure CSS)
- Responsive grid layouts that adapt from desktop to mobile
- Accessible color contrast ratios and semantic HTML throughout

### Home Page — Animated Train Background
The home page features a CSS-only animated steam train that travels across the screen from left to right, built entirely from `div` elements — no images or canvas required.

```css
/* Design system — CSS variables used across all pages */
:root {
  --coal:  #1a1410;   /* page background */
  --iron:  #2e2620;   /* card background */
  --iron2: #3a2e28;   /* input background */
  --rust:  #b94a1c;   /* primary accent */
  --gold:  #c9943a;   /* highlights */
  --steam: #e8e0d0;   /* primary text */
  --smoke: #8c7f6e;   /* secondary text */
}

/* Animated train — moves left to right across the screen */
.train-wrapper {
  position: absolute;
  bottom: 22px;
  left: 0;
  display: flex;
  align-items: flex-end;
  animation: trainLTR 22s linear infinite;
}

@keyframes trainLTR {
  0%   { transform: translateX(-900px); }
  100% { transform: translateX(calc(100vw + 100px)); }
}
```

### Responsive Design
All grid layouts use `repeat(auto-fill, minmax(...))` so they adapt automatically to screen width without media query breakpoints for every element.

```css
/* Quick Access grid — 4 columns on desktop, 2 on tablet, 1 on mobile */
.rr-quick {
  display: grid;
  grid-template-columns: repeat(4, 1fr);
  gap: 12px;
}

@media (max-width: 900px) {
  .rr-quick { grid-template-columns: repeat(2, 1fr); }
}
```

**Real problem it solved:** The original PMRR site was not mobile-friendly. Visitors at the park checking schedules on their phones had a poor experience. Our responsive design works on any screen size.

---

## 2. User Authentication

### What We Built
A complete authentication system with separate tiers for regular visitors and staff. Built with Flask sessions and SQLite — no third-party auth library.

- Visitor accounts: register, login, logout, change password
- Staff accounts: elevated permissions to view all reservations and volunteer sign-ups
- Session management with `SESSION_COOKIE_SAMESITE` configured for cross-origin requests between Jekyll (port 4500) and Flask (port 8587)
- Passwords hashed with `werkzeug.security`

```python
# api/login_api.py — registration endpoint
from flask import Blueprint, request, jsonify, session
from werkzeug.security import generate_password_hash, check_password_hash
import sqlite3, re, os

login_bp = Blueprint('login_bp', __name__, url_prefix='/api/auth')

@login_bp.route('/register', methods=['POST'])
def register():
    data = request.get_json()
    name     = data.get('name', '').strip()
    email    = data.get('email', '').strip().lower()
    password = data.get('password', '').strip()

    # Validate inputs
    if len(name) < 2:
        return jsonify({'error': 'Name must be at least 2 characters'}), 400
    if len(password) < 6:
        return jsonify({'error': 'Password must be at least 6 characters'}), 400

    try:
        with get_db() as conn:
            conn.execute(
                'INSERT INTO railroad_users (name, email, password) VALUES (?, ?, ?)',
                (name, email, generate_password_hash(password))
            )
            conn.commit()
    except sqlite3.IntegrityError:
        return jsonify({'error': 'An account with this email already exists'}), 409

    # Set session after successful registration
    session['rr_user'] = {'name': name, 'email': email}
    return jsonify({'message': 'Account created', 'name': name, 'email': email}), 201
```

```python
# Session status check — called by the header on every page load
@login_bp.route('/status', methods=['GET'])
def status():
    user = session.get('rr_user')
    if user:
        return jsonify({'logged_in': True, 'name': user['name'], 'email': user['email']}), 200
    return jsonify({'logged_in': False}), 200
```

```javascript
// header.html — checks auth status on every page and updates the nav
async function rrCheckAuth() {
    try {
        const res  = await fetch(`${RR_BACKEND}/api/auth/status`, { credentials: 'include' });
        const data = await res.json();
        if (data.logged_in) rrShowLoggedIn(data);
        else rrShowGuest();
    } catch { rrShowGuest(); }
}

function rrShowLoggedIn(user) {
    // Hide login link, show avatar with user's initial
    document.getElementById('rrLoginLink').style.display = 'none';
    document.getElementById('rrProfileItem').classList.add('visible');
    document.getElementById('rrAvatarInitial').textContent = user.name.charAt(0).toUpperCase();
    document.getElementById('rrNavName').textContent = user.name.split(' ')[0];
}
```

**Real problem it solved:** The original PMRR site had no accounts at all. Our system lets visitors track their booking history and gives staff a private dashboard to view all reservations and volunteer sign-ups without exposing that data publicly.

---

## 3. API Integration

### What We Built
The Jekyll frontend communicates with the Flask backend through a REST API. Every dynamic feature — schedules, reservations, notes, auth, forecasts — goes through a `fetch()` call with `credentials: 'include'` to maintain session cookies across the cross-origin setup.

**Endpoints we built:**

| Endpoint | Method | Purpose |
|---|---|---|
| `/api/auth/login` | POST | User login |
| `/api/auth/status` | GET | Session check |
| `/api/reservations` | GET / POST | Book a ride / list bookings |
| `/api/schedule` | GET | Live seat data by date |
| `/api/notes` | GET / POST | Community notes board |
| `/api/visitor/predict` | POST | ML visitor forecast |
| `/api/titanic/predict` | POST | Safety ML model |

```javascript
// book.md — submitting a reservation to the Flask backend
const BACKEND = 'http://localhost:8587';

async function bkSubmit() {
    const res = await fetch(`${BACKEND}/api/reservations`, {
        method:      'POST',
        credentials: 'include',          // sends session cookie cross-origin
        headers:     { 'Content-Type': 'application/json' },
        body: JSON.stringify({
            date:       bkRideDate,
            time:       bkRideTime,
            train_type: bkRideType,
            first_name: first,
            last_name:  last,
            email:      email,
            adults:     bkAdult,
            children:   bkChild,
            infants:    bkInfant
        })
    });

    const data = await res.json();

    if (!res.ok) {
        // Handle seat conflict (409) vs other errors
        if (res.status === 409 && data.available !== undefined) {
            bkShowError(`Only ${data.available} seat(s) left — reduce ticket count.`);
        } else {
            bkShowError(data.error || 'Booking failed.');
        }
        return;
    }

    // Show confirmation code to user
    document.getElementById('bkConfirmCode').textContent = data.confirm_code;
    document.getElementById('bkConfirmSection').classList.add('show');
}
```

```python
# api/reservation.py — Flask endpoint that handles the booking
@reservation_bp.route('/api/reservations', methods=['POST'])
def create_reservation():
    data = request.get_json()

    # Check seat availability before confirming
    existing = Reservation.query.filter_by(
        date=data['date'], time=data['time']
    ).all()
    booked = sum(r.adults + r.children for r in existing)
    capacity = 65 if 'Steam' in data['train_type'] else 30
    requested = data['adults'] + data['children']

    if booked + requested > capacity:
        return jsonify({
            'error': 'Not enough seats',
            'available': capacity - booked
        }), 409

    # Generate unique confirmation code
    confirm_code = 'PMR-' + ''.join(random.choices('0123456789', k=6))

    reservation = Reservation(
        confirm_code=confirm_code,
        **data
    )
    db.session.add(reservation)
    db.session.commit()

    return jsonify({'confirm_code': confirm_code, **reservation.to_dict()}), 201
```

**Real problem it solved:** The original PMRR site had zero API integration — everything was static HTML. Our frontend-backend communication enables real-time features that simply weren't possible before.

---

## 4. Database Integration

### What We Built
Three SQLite databases handle different data domains, all managed through SQLAlchemy with a `SQLALCHEMY_BINDS` configuration for separation of concerns.

| Database | Contents |
|---|---|
| `user_management.db` | Main app data — reservations, notes, likes |
| `auth.db` | Railroad user accounts |
| `volumes/` | ML training data |

Notes store images as base64 strings directly in SQLite, avoiding the need for a separate file storage service.

```python
# model/note.py — Note model with base64 image storage
from __init__ import db

class Note(db.Model):
    __tablename__ = 'notes'

    id         = db.Column(db.Integer,  primary_key=True)
    content    = db.Column(db.Text,     nullable=False)
    image_data = db.Column(db.Text,     nullable=True)   # base64 encoded
    likes      = db.Column(db.Integer,  default=0)
    author     = db.Column(db.String(100), nullable=True)
    created_at = db.Column(db.DateTime, server_default=db.func.now())

    def to_dict(self):
        return {
            'id':         self.id,
            'content':    self.content,
            'image_data': self.image_data,
            'likes':      self.likes,
            'author':     self.author,
            'created_at': str(self.created_at)
        }
```

```python
# __init__.py — SQLite bind configuration
app.config['SQLALCHEMY_DATABASE_URI'] = 'sqlite:///volumes/user_management.db'
app.config['SQLALCHEMY_BINDS'] = {
    'auth': 'sqlite:///volumes/auth.db'
}

db = SQLAlchemy(app)
```

**Real problem it solved:** Every note, reservation, and user account persists across sessions. Visitors can come back the next day and their booking history is still there — something impossible with the original static site.

---

## 5. Machine Learning

### What We Built
A Gradient Boosting Regressor that predicts hourly visitor counts at the railroad. The model achieves **R² = 0.93** on test data.

**Features used:**
- `month`, `day_of_month`, `day_of_week`
- `is_saturday`, `is_holiday`, `is_school_break`
- `temperature`, `temp_bucket`
- `has_event`, `capacity`, `rides`
- One-hot encoded `weather` and `train_type`

The frontend auto-fetches tomorrow's weather from the Open-Meteo API and auto-detects Poway USD school breaks and US holidays before calling the prediction endpoint.

In [ ]:
# model/visitor.py — ML model training (simplified)
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

# Feature engineering with pandas
def build_features(df):
    df = df.copy()
    df['month']          = pd.to_datetime(df['date']).dt.month
    df['day_of_month']   = pd.to_datetime(df['date']).dt.day
    df['is_saturday']    = (pd.to_datetime(df['date']).dt.dayofweek == 5).astype(int)
    df['temp_bucket']    = pd.cut(df['temperature'], bins=[0,60,75,90,120],
                                   labels=[0,1,2,3]).astype(int)

    # One-hot encode weather and train type
    df = pd.get_dummies(df, columns=['weather', 'train_type'])
    return df

# Train the model
FEATURES = ['month','day_of_month','is_saturday','is_holiday',
            'is_school_break','has_event','temperature','temp_bucket',
            'capacity','rides']

df = build_features(pd.read_csv('data/visitor_data.csv'))
X, y = df[FEATURES], df['visitors']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.08,
    max_depth=4,
    random_state=42
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(f"R² Score: {r2_score(y_test, y_pred):.3f}")  # 0.930

**Key insight:** The model currently uses weather and calendar data as proxies because we don't have access to real PMRR attendance records. With actual historical ridership data from a formal partnership, model accuracy would improve substantially — the current R²=0.93 represents what's achievable from external signals alone.

---

## 6. AI Integration

### What We Built
A floating AI visitor assistant available on every page of the site. The widget appears as a fixed button in the bottom-right corner and opens a chat interface where visitors can ask about visiting, booking, history, or volunteering.

The assistant uses an intent-matching system that maps visitor queries to relevant pages and recommendations. It can also directly navigate users to the right page based on their question.

```javascript
// assistant.md — intent-based response engine
function getReply(prompt) {
    const p = prompt.toLowerCase();

    if (p.includes('family') || p.includes('kids') || p.includes('children'))
        return {
            text: 'For families: Start with the Steam Locomotive ride (10am–2pm, Saturdays). '
                + 'Kids under 2 ride free! Book ahead to guarantee seats.',
            link: '/railroad/book',
            linkText: 'Book a Ride →'
        };

    if (p.includes('history') || p.includes('learn') || p.includes('baldwin'))
        return {
            text: 'Explore our interactive history timeline — including the true story '
                + 'of how our 1907 Baldwin locomotive had its identity hidden for decades.',
            link: '/railroad/history',
            linkText: 'View History →'
        };

    if (p.includes('volunteer') || p.includes('help') || p.includes('sign up'))
        return {
            text: 'We always need volunteers! View open shifts and sign up directly '
                + 'on our volunteer schedule page.',
            link: '/railroad/volunteer-schedule',
            linkText: 'View Shifts →'
        };

    if (p.includes('busy') || p.includes('crowd') || p.includes('forecast'))
        return {
            text: 'Our ML-powered forecast predicts visitor numbers based on weather, '
                + 'holidays, and school schedules — plan your visit around peak times.',
            link: '/railroad/forecast',
            linkText: 'Check Forecast →'
        };

    return {
        text: 'I can help with visit planning, booking, history, or volunteering. '
            + 'Try asking: "best time to visit with kids" or "how do I volunteer?"',
        link: null
    };
}
```

**Real problem it solved:** The original PMRR site had no help system. Basic questions about hours, prices, and booking required visitors to call or email volunteers. Our assistant handles these 24/7 with no staff required.

**Future upgrade:** Connecting to a live LLM (like Claude or GPT) with real park data as context would allow the assistant to answer complex, open-ended questions far beyond what our current intent-matching can handle.

---

## 7. Social Messaging

### What We Built
A community visitor notes board where anyone can post text and photos from their visit. Posts include a like system and persist in the database across sessions.

The contact page also includes a structured form for reaching the railroad directly.

```javascript
// notes.md — submitting a new note with optional image
async function submitNote() {
    const content = document.getElementById('noteContent').value.trim();
    const file    = document.getElementById('noteImage').files[0];

    let image_data = null;

    // Convert image to base64 if one was uploaded
    if (file) {
        image_data = await new Promise((resolve) => {
            const reader = new FileReader();
            reader.onload = () => resolve(reader.result);
            reader.readAsDataURL(file);
        });
    }

    const res = await fetch(`${BACKEND}/api/notes`, {
        method:      'POST',
        credentials: 'include',
        headers:     { 'Content-Type': 'application/json' },
        body:        JSON.stringify({ content, image_data })
    });

    if (res.ok) loadNotes();  // refresh the board
}
```

```python
# api/note_api.py — like endpoint
@note_bp.route('/api/notes/<int:note_id>/like', methods=['POST'])
def like_note(note_id):
    note = Note.query.get_or_404(note_id)
    note.likes += 1
    db.session.commit()
    return jsonify({'likes': note.likes}), 200
```

**Real problem it solved:** The original PMRR site had zero community or social features. Visitors had no way to share their experience online. Our notes board creates a living archive of visitor memories and generates organic content that promotes the railroad.

---

## 8. Blogging / CMS

### What We Built
The entire site is built on Jekyll, a static site generator that uses Markdown files and Liquid templates for content management. This makes it easy to add new pages, update schedules, and publish announcements without touching HTML directly.

The Events & Announcements page acts as a lightweight CMS — structured YAML data drives the content, which is filtered and searched in real time by the frontend.

```yaml
# Jekyll front matter — every page is managed this way
---
layout: base
title: Railroad History
permalink: /railroad/history
---
```

```javascript
// events.md — dynamic content filtering without page reload
const UPDATES = [
    { id:1, title: "Spring Family Ride Window Expanded",
      category: "events", date: "2026-03-12",
      summary: "Saturday rides now run until 4:30 PM during spring programming weeks." },
    { id:2, title: "Volunteer Engineer Orientation",
      category: "volunteer", date: "2026-03-08",
      summary: "New volunteer orientation covers dispatch basics and safety checks." },
    // ...
];

function render() {
    const q      = searchEl.value.trim().toLowerCase();
    const cat    = catEl.value;

    const filtered = UPDATES.filter(u => {
        const matchQ   = !q || (u.title + ' ' + u.summary).toLowerCase().includes(q);
        const matchCat = cat === 'all' || u.category === cat;
        return matchQ && matchCat;
    });

    statusEl.textContent = `${filtered.length} item(s) shown`;
    // render filtered results...
}
```

**Real problem it solved:** The original PMRR site required direct HTML editing to update event listings. Our Jekyll-based system means new content can be added by editing a structured data file — no web development knowledge required.

---

## 9. Gamification

### What We Built
The volunteer scheduling portal incorporates gamification mechanics — volunteers can see their sign-up count, track open spots, and earn a visual "signed" status badge when they commit to a shift. The live stats dashboard (total shifts, open spots, signed crew) creates a sense of collective progress.

The visitor forecast page also uses interactive gamification-adjacent elements — visitors can select time slots, adjust parameters, and watch the prediction update in real time.

```javascript
// volunteer-schedule.md — sign-up with live feedback
function signUpForShift(iso) {
    if (!volunteerData[iso]) volunteerData[iso] = [];
    const slots    = volunteerData[iso];
    const shiftObj = allOps.find(op => op.iso === iso);

    // Prevent over-signing
    if (slots.length >= shiftObj.slotsTotal) return false;
    if (slots.includes(currentUser))         return false;

    slots.push(currentUser);
    saveVolunteerData();
    return true;  // triggers table re-render with updated stats
}

// Live stats update after every sign-up / cancellation
function updateStats() {
    let openSpotsCount = 0, totalSigned = 0;
    for (let op of allOps) {
        const taken = volunteerData[op.iso]?.length || 0;
        openSpotsCount += Math.max(0, op.slotsTotal - taken);
        totalSigned    += taken;
    }
    document.getElementById('openSpots').innerText    = openSpotsCount;
    document.getElementById('signedUpCount').innerText = totalSigned;
}
```

**Real problem it solved:** Manual volunteer coordination had no visibility into how many shifts were covered. The live stats dashboard turns shift coverage into a collective goal — volunteers can see at a glance how many spots are still open and feel motivated to fill them.

---

## Reflection

Building for a real organization changed how I approached every decision. Three things I'll carry forward:

**Design for the moment before the visit.** A visitor deciding whether to drive to the park on a Saturday morning needs accurate information before they leave home — not after they arrive. That framing changed how we prioritized every feature.

**Proxy data has a ceiling.** Our ML forecast performs well, but it was trained on weather and calendar signals because we didn't have access to real attendance records. The gap between what a model *can* do and what it *could* do with the right data is something I want to keep thinking about.

**Architecture is product.** Every early backend decision — session handling, CORS configuration, API structure — had downstream effects on what the frontend could or couldn't do. The two aren't separate concerns.

---

**Project links:**
- Frontend: [rcr.opencodingsociety.com](https://rcr.opencodingsociety.com)
- GitHub (Frontend): [RCR-Dominators-Frontend](https://github.com/Rishabh-da-best/RCR-Dominators-Frontend-)
- Original PMRR site: [powaymidlandrr.org](https://powaymidlandrr.org)